# 82 — Giải phẫu một Strategy

Notebook [`81`](81_tu_finlens_toi_cerebro.ipynb) chạy được chiến lược đầu tiên.
Notebook này mở nó ra: một `bt.Strategy` thực chất là gì, mỗi phương thức được
gọi lúc nào, lệnh đi qua những trạng thái nào, và **bốn loại lệnh khớp ở đâu** —
mỗi loại kiểm bằng tay trên một phiên thật.

| Cạm bẫy | Triệu chứng | Đo bằng cách |
|---|---|---|
| **`close[1]` đọc được tương lai** | không lỗi; chiến lược "biết trước" giá mai | so với `close[0]` của phiên sau |
| **Cheat-on-close** (`set_coc(True)`) | lệnh thị trường khớp ở giá đóng cửa phiên tín hiệu | chạy cùng chiến lược bật/tắt trên 15 mã, đo chênh CAGR |

In [1]:
import datetime as dt
import sys
from collections import Counter
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import backtrader as bt
import numpy as np
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, hom_nay, lui_ngay
from finlens_examples import bt_vn
from finlens_examples.charts import GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


## 1 · Dữ liệu

Năm năm giá ngày của HPG, đưa qua đúng hai hàm đã dựng tay ở notebook `81`:
bỏ phiên dở dang, rồi quy về VND. Giữ lại `khung` (frame pandas cùng nội dung với
feed) để kiểm mọi con số backtrader in ra.

In [2]:
MA = "HPG"
VON = 100_000_000
PHI = 0.0015

gia = bt_vn.bo_phien_chua_dong(client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=5)), client)
khung = bt_vn.chuan_bi_frame(gia)  # cùng dữ liệu với feed, để đối chiếu


def cerebro_moi(**kw):
    """Cerebro một mã, vốn 100 triệu, phí 0,15%/chiều. ``kw`` chuyển cho Cerebro."""
    cerebro = bt.Cerebro(stdstats=False, **kw)
    cerebro.adddata(bt_vn.feed_tu_finlens(gia))
    cerebro.broker.setcash(VON)
    cerebro.broker.setcommission(commission=PHI)
    return cerebro


print(f"{MA}: {len(khung)} phiên · {khung.index[0]:%d/%m/%Y} → {khung.index[-1]:%d/%m/%Y} · giá VND")

HPG: 1245 phiên · 20/10/2021 → 19/10/2026 · giá VND


## 2 · `lines` và chỉ số `[0]`, `[-1]`

Trong backtrader mọi chuỗi là một **line**: `data.close`, `data.open`, một SMA,
cả phép so sánh giữa hai line. Chỉ số không đếm từ đầu chuỗi như pandas mà đếm
**tương đối so với phiên đang xử lý**:

| Viết | Nghĩa | Tương đương pandas ở phiên thứ `n` |
|---|---|---|
| `close[0]` | phiên hiện tại | `close.iloc[n-1]` |
| `close[-1]` | phiên trước | `close.iloc[n-2]` |
| `len(self)` | số phiên đã đi qua | `n` |

Kiểm ở ba phiên, so với `khung`:

In [3]:
class SoiLines(bt.Strategy):
    def __init__(self):
        self.sma5 = bt.ind.SMA(self.data.close, period=5)
        self.tren_sma = self.data.close > self.sma5  # một line, chưa có giá trị nào
        print(f"__init__: sma5 là {type(self.sma5).__name__}, tren_sma là {type(self.tren_sma).__name__}")
        self.ghi = []

    def next(self):
        if len(self) in (5, 6, 200):
            self.ghi.append(
                {
                    "len(self)": len(self),
                    "ngày": self.data.datetime.date(0),
                    "close[0]": self.data.close[0],
                    "close[-1]": self.data.close[-1],
                    "sma5[0]": self.sma5[0],
                    "tren_sma[0]": self.tren_sma[0],
                }
            )


cerebro = cerebro_moi()
cerebro.addstrategy(SoiLines)
soi = pd.DataFrame(cerebro.run()[0].ghi).set_index("len(self)")

sma5_pd = khung["close"].rolling(5).mean()
soi["pandas close.iloc[n-1]"] = [khung["close"].iloc[n - 1] for n in soi.index]
soi["pandas close.iloc[n-2]"] = [khung["close"].iloc[n - 2] for n in soi.index]
soi["pandas sma5.iloc[n-1]"] = [sma5_pd.iloc[n - 1] for n in soi.index]
soi

__init__: sma5 là SMA, tren_sma là LinesOperation


,ngày,close[0],close[-1],sma5[0],tren_sma[0],pandas close.iloc[n-1],pandas close.iloc[n-2],pandas sma5.iloc[n-1]
len(self),,,,,,,,
5,2021-10-26,28820.0,28200.0,28890.0,0.0,28820.0,28200.0,28890.0
6,2021-10-27,29640.0,28820.0,28992.0,1.0,29640.0,28820.0,28992.0
200,2022-08-08,15960.0,15760.0,15854.0,1.0,15960.0,15760.0,15854.0


Ba cột cuối (pandas) trùng với ba cột backtrader tương ứng. `tren_sma[0]` là
`1.0`/`0.0` chứ không phải `True`/`False` — line chỉ chứa số thực.

### Chỉ số dương đọc tương lai — và backtrader không chặn

`close[-1]` là hôm qua thì `close[1]` là ngày mai. Mặc định backtrader **nạp sẵn
toàn bộ dữ liệu vào bộ nhớ** (`preload=True`) để chạy nhanh, nên `close[1]` trả
về giá đóng cửa của phiên sau — không lỗi, không cảnh báo.

In [4]:
class NhinTruoc(bt.Strategy):
    def __init__(self):
        self.ghi = []

    def next(self):
        if len(self) in (100, 101):
            try:
                self.ghi.append((len(self), self.data.close[0], self.data.close[1]))
            except IndexError as loi:
                self.ghi.append((len(self), self.data.close[0], f"IndexError: {loi}"))


for che_do in [dict(), dict(preload=False)]:
    cerebro = cerebro_moi(**che_do)
    cerebro.addstrategy(NhinTruoc)
    ghi = cerebro.run()[0].ghi
    print(f"{str(che_do) or 'mặc định':<20}", " · ".join(f"len={n}: close[0]={c0:,.0f} close[1]={c1}" for n, c0, c1 in ghi))

print(f"\nkhung: phiên 100 = {khung['close'].iloc[99]:,.0f} · phiên 101 = {khung['close'].iloc[100]:,.0f}")

{}                   len=100: close[0]=23,650 close[1]=23520.0 · len=101: close[0]=23,520 close[1]=23800.0


{'preload': False}   len=100: close[0]=23,650 close[1]=IndexError: array index out of range · len=101: close[0]=23,520 close[1]=IndexError: array index out of range

khung: phiên 100 = 23,650 · phiên 101 = 23,520


Ở chế độ mặc định, `close[1]` tại phiên 100 chính là `close[0]` của phiên 101 —
**một lỗi nhìn trước gõ thiếu dấu trừ**. Tắt `preload` thì backtrader mới ném
`IndexError`. Không cần chạy như vậy mọi lúc (chậm hơn), nhưng đáng chạy một lần
mỗi khi viết chiến lược mới: nếu nó vỡ, có chỗ đang đọc tương lai.

## 3 · `__init__` và `next`

Hai nơi viết logic, hai cách tính khác nhau:

| | `__init__` | `next` |
|---|---|---|
| Chạy | **một lần**, trước phiên đầu tiên | **mỗi phiên** một lần |
| `close > sma` cho ra | một **line** (`LinesOperation`), tính cả chuỗi một lượt | một `bool` của phiên hiện tại |
| Hợp với | chỉ báo, điều kiện cố định | quyết định phụ thuộc trạng thái: đang giữ hàng không, tiền còn bao nhiêu |

Cùng một điều kiện viết hai nơi phải cho cùng kết quả ở mọi phiên:

In [5]:
class HaiNoi(bt.Strategy):
    def __init__(self):
        self.sma20 = bt.ind.SMA(self.data.close, period=20)
        self.tren_init = self.data.close > self.sma20  # line, tính một lần
        self.lech = 0
        self.so_phien = 0

    def next(self):
        tren_next = self.data.close[0] > self.sma20[0]  # bool, tính mỗi phiên
        self.lech += bool(self.tren_init[0]) != tren_next
        self.so_phien += 1


cerebro = cerebro_moi()
cerebro.addstrategy(HaiNoi)
hai_noi = cerebro.run()[0]
tren_pd = (khung["close"] > khung["close"].rolling(20).mean()).iloc[19:]
print(f"{hai_noi.so_phien} phiên được gọi next · lệch giữa __init__ và next: {hai_noi.lech}")
print(f"pandas: {len(tren_pd)} phiên có SMA20 · {tren_pd.sum()} phiên trên SMA20")

1226 phiên được gọi next · lệch giữa __init__ và next: 0
pandas: 1226 phiên có SMA20 · 591 phiên trên SMA20


## 4 · `minperiod` và `prenext`

`next` chỉ được gọi khi **mọi** chỉ báo đã có giá trị. SMA50 cần 50 phiên; SMA20
*của* SMA50 cần thêm 19 phiên nữa. backtrader tự cộng các khoảng khởi động đó
thành `minperiod` của strategy, và trước mốc ấy gọi `prenext` thay cho `next`.

In [6]:
class DemPhien(bt.Strategy):
    def __init__(self):
        self.sma50 = bt.ind.SMA(self.data.close, period=50)
        self.sma_cua_sma = bt.ind.SMA(self.sma50, period=20)
        self.dem = Counter()
        self.phien_next_dau = None

    def prenext(self):
        self.dem["prenext"] += 1

    def nextstart(self):  # gọi đúng một lần, ở phiên next đầu tiên
        self.phien_next_dau = (len(self), self.data.datetime.date(0))
        super().nextstart()  # mặc định nextstart() gọi next()

    def next(self):
        self.dem["next"] += 1


cerebro = cerebro_moi()
cerebro.addstrategy(DemPhien)
dem = cerebro.run()[0]
print(f"minperiod: sma50 = {dem.sma50._minperiod} · sma20(sma50) = {dem.sma_cua_sma._minperiod} · strategy = {dem._minperiod}")
print(f"prenext gọi {dem.dem['prenext']} lần · next gọi {dem.dem['next']} lần · tổng {sum(dem.dem.values())} = {len(khung)} phiên")
print(f"next đầu tiên ở phiên thứ {dem.phien_next_dau[0]} ({dem.phien_next_dau[1]:%d/%m/%Y})")

minperiod: sma50 = 50 · sma20(sma50) = 69 · strategy = 69
prenext gọi 68 lần · next gọi 1177 lần · tổng 1245 = 1245 phiên
next đầu tiên ở phiên thứ 69 (25/01/2022)


Hai hệ quả thực tế:

- **Chiến lược dùng chỉ báo dài "mất" phần đầu dữ liệu.** 68 phiên ở trên là
  hơn ba tháng không thể có giao dịch. So sánh hai chiến lược khác `minperiod`
  trên cùng khoảng thời gian là so hai khoảng thời gian khác nhau.
- **Nhiều mã trong một Cerebro** thì `next` chờ mã **chậm nhất** — một mã niêm
  yết muộn làm cả danh mục đứng chờ. Đó là việc của `prenext`, sẽ gặp lại ở
  notebook `86`.

## 5 · `notify_order`, `notify_trade` và nhật ký giao dịch

Mỗi lệnh đi qua một chuỗi trạng thái; broker báo từng bước qua `notify_order`.
Khi một **vị thế** mở rồi đóng, broker báo qua `notify_trade` — một *trade* là
cả vòng mua-bán, có lãi/lỗ trước và sau phí.

| Trạng thái | Nghĩa |
|---|---|
| `Submitted` → `Accepted` | broker nhận lệnh, chờ phiên sau |
| `Completed` | khớp toàn bộ |
| `Margin` | **không đủ tiền** lúc khớp — lệnh bị huỷ |
| `Canceled` / `Expired` | huỷ tay / hết hiệu lực |

In [7]:
class GiaoCatSMA(bt.Strategy):
    """Giao cắt SMA 20/50 của notebook 81, thêm nhật ký đầy đủ."""

    params = (("nhanh", 20), ("cham", 50), ("ty_le", 0.95))

    def __init__(self):
        self.cat = bt.ind.CrossOver(bt.ind.SMA(period=self.p.nhanh), bt.ind.SMA(period=self.p.cham))
        self.lenh_cho = None
        self.nk_lenh, self.nk_trade = [], []

    def next(self):
        if self.lenh_cho:  # còn lệnh chưa xử lý xong thì không đặt thêm
            return
        if not self.position and self.cat[0] > 0:
            self.lenh_cho = self.buy(size=int(self.broker.getcash() * self.p.ty_le / self.data.close[0]))
        elif self.position and self.cat[0] < 0:
            self.lenh_cho = self.close()

    def notify_order(self, order):
        xong = order.status == order.Completed
        self.nk_lenh.append(
            {
                "ngay_dat": bt.num2date(order.created.dt).date(),
                "ngay": bt.num2date(order.executed.dt).date() if xong else self.data.datetime.date(0),
                "ref": order.ref,
                "chieu": "mua" if order.isbuy() else "bán",
                "trang_thai": order.getstatusname(),
                "so_cp": order.executed.size if xong else order.created.size,
                "gia": order.executed.price if xong else np.nan,
                "pclose_luc_dat": order.created.pclose,
            }
        )
        if not order.alive():
            self.lenh_cho = None

    def notify_trade(self, trade):
        if trade.isclosed:
            self.nk_trade.append(
                {
                    "mo": bt.num2date(trade.dtopen).date(),
                    "dong": bt.num2date(trade.dtclose).date(),
                    "so_phien": trade.barlen,
                    "gia_vao": trade.price,
                    "lai_lo": trade.pnl,
                    "lai_lo_sau_phi": trade.pnlcomm,
                }
            )


cerebro = cerebro_moi()
cerebro.addstrategy(GiaoCatSMA)
gc = cerebro.run()[0]
nk_lenh = pd.DataFrame(gc.nk_lenh)
nk_trade = pd.DataFrame(gc.nk_trade)
print(f"{len(nk_lenh)} thông báo lệnh · {len(nk_trade)} vòng mua-bán đã đóng")
nk_lenh.head(6)

84 thông báo lệnh · 14 vòng mua-bán đã đóng


,ngay_dat,ngay,ref,chieu,trang_thai,so_cp,gia,pclose_luc_dat
0,2022-03-01,2022-03-02,1,mua,Submitted,3956,NaN,24010.0
1,2022-03-01,2022-03-02,1,mua,Accepted,3956,NaN,24010.0
2,2022-03-01,2022-03-02,1,mua,Completed,3956,24060.0,24010.0
3,2022-04-07,2022-04-08,2,bán,Submitted,-3956,NaN,23830.0
4,2022-04-07,2022-04-08,2,bán,Accepted,-3956,NaN,23830.0
5,2022-04-07,2022-04-08,2,bán,Completed,-3956,23930.0,23830.0


Mỗi lệnh sinh **ba** thông báo, cả ba tới trong phiên *sau* phiên đặt lệnh
(`ngay_dat`): backtrader chuyển lệnh cho broker ở đầu vòng lặp kế tiếp, broker
nhận rồi khớp luôn ở giá mở cửa. `pclose_luc_dat` là giá đóng cửa của phiên tín
hiệu — chỗ tham chiếu để thấy lệnh thị trường khớp ở một giá **khác**.

Nhật ký vòng mua-bán:

In [8]:
tong = nk_trade[["lai_lo", "lai_lo_sau_phi"]].sum()
print(f"Tổng lãi/lỗ trước phí {tong['lai_lo']:,.0f} · sau phí {tong['lai_lo_sau_phi']:,.0f} · phí {tong['lai_lo'] - tong['lai_lo_sau_phi']:,.0f} VND")
print(f"Thắng {(nk_trade['lai_lo_sau_phi'] > 0).sum()}/{len(nk_trade)} vòng · giữ trung bình {nk_trade['so_phien'].mean():.0f} phiên")
nk_trade.style.format({"gia_vao": "{:,.0f}", "lai_lo": "{:,.0f}", "lai_lo_sau_phi": "{:,.0f}"})

Tổng lãi/lỗ trước phí 1,114,340 · sau phí -3,255,825 · phí 4,370,165 VND
Thắng 4/14 vòng · giữ trung bình 44 phiên


,mo,dong,so_phien,gia_vao,lai_lo,lai_lo_sau_phi
0,2022-03-02,2022-04-08,27,"24,060","-514,280","-799,053"
1,2022-08-16,2022-09-29,30,"16,230","-7,257,500","-7,529,308"
2,2022-12-13,2023-03-23,66,"12,580","7,475,760","7,203,310"
3,2023-04-18,2023-08-31,94,"14,140","29,241,790","28,915,368"
4,2023-09-22,2023-10-05,9,"18,770","-10,047,210","-10,387,962"
5,2023-11-24,2024-04-23,100,"17,580","8,676,800","8,327,304"
6,2024-05-23,2024-07-17,39,"21,460","-1,241,240","-1,602,610"
7,2024-10-10,2024-11-28,35,"20,530","-4,898,550","-5,246,145"
8,2024-12-18,2025-01-13,17,"20,090","-6,070,680","-6,400,352"
9,2025-02-28,2025-04-10,28,"21,130","-21,197,400","-21,485,533"


### Trạng thái `Margin`: vì sao chỉ dùng 95% tiền

Khối lượng tính bằng giá **đóng cửa** hôm nay, nhưng lệnh khớp ở giá **mở cửa**
phiên sau, và còn phải trả phí. Tiền không đủ lúc khớp thì broker huỷ lệnh —
lặng lẽ, chỉ để lại một dòng `Margin` trong `notify_order`. Thử bốn mức:

In [9]:
ket = []
for ty_le in [1.0, 0.998, 0.99, 0.95]:
    cerebro = cerebro_moi()
    cerebro.addstrategy(GiaoCatSMA, ty_le=ty_le)
    s = cerebro.run()[0]
    tt = pd.DataFrame(s.nk_lenh)["trang_thai"].value_counts()
    ket.append(
        {
            "tỷ lệ tiền": f"{ty_le:.1%}",
            "lệnh khớp": tt.get("Completed", 0),
            "lệnh bị huỷ Margin": tt.get("Margin", 0),
            "giá trị cuối kỳ": f"{cerebro.broker.getvalue():,.0f}",
        }
    )
pd.DataFrame(ket).set_index("tỷ lệ tiền")

,lệnh khớp,lệnh bị huỷ Margin,giá trị cuối kỳ
tỷ lệ tiền,,,
100.0%,0,14,"100,000,000"
99.8%,12,8,"67,359,630"
99.0%,24,2,"105,967,577"
95.0%,28,0,"96,744,175"


- **100%**: *mọi* lệnh mua bị huỷ — kể cả khi giá mở cửa bằng giá đóng cửa, phí
  0,15% đã làm thiếu tiền. Chiến lược không bao giờ giao dịch, và giá trị cuối kỳ
  bằng đúng vốn ban đầu. Một backtest như vậy trông như "chiến lược an toàn,
  không lỗ".
- **99,8%**: đủ phí, nhưng lệnh vẫn bị huỷ mỗi khi giá mở cửa phiên sau cao hơn
  giá đóng cửa phiên tín hiệu quá phần dư — 8 lần trong 5 năm.
- **99%**: còn 2 lần bị huỷ.
- **95%**: không lệnh nào bị huỷ.

Để ý cột cuối: giá trị cuối kỳ nhảy theo một tham số thuần kỹ thuật, và bản 99%
còn "lãi" hơn bản 95% — không phải vì nó tốt hơn, mà vì nó bỏ lỡ một tập vòng
giao dịch khác. Đó là kết quả của một sự cố, không phải của chiến lược.

Mỗi dòng `Margin` là một tín hiệu mua **không được thực hiện** — backtest bỏ lỡ
cả một vòng giao dịch mà con số cuối không nói gì. Đây là lý do `bt_vn.LoVN`
chừa 5% (notebook [`83`](83_luat_choi_viet_nam.ipynb)).

## 6 · Bốn loại lệnh, mỗi loại một ví dụ có số

| Loại | Đặt | Khớp khi | Giá khớp (lệnh mua) |
|---|---|---|---|
| `Market` | không giá | phiên kế tiếp | giá mở cửa |
| `Limit` | giá tối đa chịu trả | giá mở cửa ≤ giới hạn, hoặc giá thấp nhất chạm giới hạn | `min(mở cửa, giới hạn)` |
| `Stop` | giá kích hoạt | giá mở cửa ≥ kích hoạt, hoặc giá cao nhất chạm kích hoạt | `max(mở cửa, kích hoạt)` |
| `StopTrail` (bán) | % bám đuổi | giá chạm mức dừng — mức này **chỉ đi lên**, theo giá đóng cửa | `min(mở cửa, mức dừng)` |

Cả bốn đặt ở **cùng một phiên**: phiên tín hiệu mua đầu tiên của chiến lược giao
cắt ở mục 5. Mỗi kết quả của backtrader được đối chiếu với một bộ khớp lệnh viết
tay bằng pandas — nếu hai bên lệch nhau, bảng sẽ báo.

In [10]:
MOC = nk_lenh[nk_lenh["chieu"] == "mua"]["ngay_dat"].iloc[0]  # phiên tín hiệu mua đầu tiên
SO_CP = 1_000
HIEU_LUC = dt.timedelta(days=10)  # Limit/Stop hết hạn sau 10 ngày lịch


class ThuLenh(bt.Strategy):
    """Đặt đúng một lệnh mua ở phiên ``moc``; ``trail`` thì kèm lệnh bán StopTrail sau khi mua khớp."""

    params = (("moc", None), ("loai", "Market"), ("lech", 0.0), ("trail", None))

    def __init__(self):
        self.ket_qua, self.muc_dung = {}, []
        self.lenh_ban = None

    def next(self):
        hom_nay = self.data.datetime.date(0)
        if hom_nay == self.p.moc:
            gia = self.data.close[0] * (1 + self.p.lech)
            kieu = getattr(bt.Order, self.p.loai)
            kw = {} if self.p.loai == "Market" else {"price": gia, "valid": hom_nay + HIEU_LUC}
            self.buy(size=SO_CP, exectype=kieu, **kw)
            self.ket_qua["gia_dat"] = gia if self.p.loai != "Market" else np.nan
        if self.lenh_ban is not None and self.lenh_ban.alive():
            self.muc_dung.append((hom_nay, self.lenh_ban.created.price))  # mức dừng áp cho phiên sau

    def notify_order(self, order):
        if order.status in (order.Completed, order.Expired, order.Canceled, order.Margin):
            khoa = "mua" if order.isbuy() else "ban"
            self.ket_qua[f"{khoa}_trang_thai"] = order.getstatusname()
            self.ket_qua[f"{khoa}_ngay"] = self.data.datetime.date(0)
            self.ket_qua[f"{khoa}_gia"] = order.executed.price if order.status == order.Completed else np.nan
            if order.status == order.Completed and order.isbuy() and self.p.trail:
                self.lenh_ban = self.sell(size=SO_CP, exectype=bt.Order.StopTrail, trailpercent=self.p.trail)


def chay_lenh(**tham_so):
    cerebro = cerebro_moi()
    cerebro.addstrategy(ThuLenh, moc=MOC, **tham_so)
    return cerebro.run()[0]


print(f"Phiên đặt lệnh: {MOC:%d/%m/%Y} · giá đóng cửa {khung.loc[pd.Timestamp(MOC), 'close']:,.0f} VND")

Phiên đặt lệnh: 01/03/2022 · giá đóng cửa 24,010 VND


Bộ khớp lệnh tay — đúng các quy tắc trong bảng trên, đi từng phiên sau phiên đặt
lệnh:

In [11]:
def khop_tay(loai, gia_dat, i_dat, het_han):
    """Phiên và giá khớp của lệnh MUA ``loai`` đặt ở phiên thứ ``i_dat`` của ``khung``."""
    for k in range(i_dat + 1, len(khung)):
        ngay, o, h, l = khung.index[k], *khung[["open", "high", "low"]].iloc[k]
        if loai != "Market" and ngay.date() > het_han:
            return None, np.nan
        if loai == "Market":
            return ngay.date(), o
        if loai == "Limit" and (o <= gia_dat or l <= gia_dat):
            return ngay.date(), min(o, gia_dat)
        if loai == "Stop" and (o >= gia_dat or h >= gia_dat):
            return ngay.date(), max(o, gia_dat)
    return None, np.nan


i_moc = khung.index.get_loc(pd.Timestamp(MOC))
dong = []
for loai, lech in [("Market", 0.0), ("Limit", -0.02), ("Stop", +0.02)]:
    s = chay_lenh(loai=loai, lech=lech)
    kq = s.ket_qua
    gia_dat = kq["gia_dat"] if loai != "Market" else np.nan
    ngay_tay, gia_tay = khop_tay(loai, gia_dat, i_moc, MOC + HIEU_LUC)
    ohlc = khung.loc[pd.Timestamp(kq["mua_ngay"]), ["open", "high", "low"]] if kq["mua_trang_thai"] == "Completed" else None
    dong.append(
        {
            "loại": f"{loai} {lech:+.0%}" if lech else loai,
            "giá đặt": gia_dat,
            "trạng thái": kq["mua_trang_thai"],
            "ngày (bt)": kq["mua_ngay"],
            "giá khớp (bt)": kq["mua_gia"],
            "mở cửa": ohlc["open"] if ohlc is not None else np.nan,
            "cao nhất": ohlc["high"] if ohlc is not None else np.nan,
            "thấp nhất": ohlc["low"] if ohlc is not None else np.nan,
            "ngày (tay)": ngay_tay,
            "giá khớp (tay)": gia_tay,
        }
    )
bang_lenh = pd.DataFrame(dong).set_index("loại")
bang_lenh["khớp tay = bt"] = (bang_lenh["giá khớp (tay)"] == bang_lenh["giá khớp (bt)"]) | (
    bang_lenh["giá khớp (tay)"].isna() & bang_lenh["giá khớp (bt)"].isna()
)
bang_lenh.style.format(
    {c: "{:,.0f}" for c in ["giá đặt", "giá khớp (bt)", "mở cửa", "cao nhất", "thấp nhất", "giá khớp (tay)"]}, na_rep="—"
)

,giá đặt,trạng thái,ngày (bt),giá khớp (bt),mở cửa,cao nhất,thấp nhất,ngày (tay),giá khớp (tay),khớp tay = bt
loại,,,,,,,,,,
Market,—,Completed,2022-03-02,"24,060","24,060","24,240","23,850",2022-03-02,"24,060",True
Limit -2%,"23,530",Expired,2022-03-14,—,—,—,—,—,—,True
Stop +2%,"24,490",Completed,2022-03-03,"24,490","24,370","25,640","24,210",2022-03-03,"24,490",True


Đọc từng dòng theo cột "mở cửa / cao nhất / thấp nhất" của phiên khớp:

- **Market** khớp ở giá mở cửa phiên ngay sau phiên đặt — không bàn cãi.
- **Limit −2%** chỉ khớp khi giá lùi về tới mức giới hạn trong thời hạn hiệu
  lực. Nếu giá mở cửa đã thấp hơn giới hạn, lệnh khớp ở giá mở cửa — **tốt hơn**
  giá đặt, đúng như một lệnh giới hạn thật. Ở lần chạy lưu trong file này, giá
  không lùi về mức đó trong 10 ngày nên lệnh `Expired` — bộ tay cũng không tìm
  thấy phiên khớp. Một lệnh giới hạn **có thể không bao giờ khớp**, và backtest
  phải tính cả những lần bỏ lỡ đó.
- **Stop +2%** là lệnh "mua khi giá xác nhận đi lên". Nếu phiên mở cửa vọt qua
  mức kích hoạt, nó khớp ở giá mở cửa — **xấu hơn** giá đặt. Đó là trượt giá
  của lệnh dừng, và backtrader mô hình hoá đúng điều đó.

Cột cuối cùng là phép kiểm: bộ khớp tay và backtrader cho cùng ngày, cùng giá.

### `StopTrail`: mức dừng bám theo giá

Mua bằng lệnh Market ở cùng phiên, rồi ngay khi khớp đặt lệnh bán `StopTrail`
cách 5%. Mức dừng khởi đầu = 95% giá đóng cửa phiên khớp, sau mỗi phiên chưa
chạm thì nâng lên 95% giá đóng cửa nếu cao hơn — **không bao giờ hạ xuống**.

In [12]:
TRAIL = 0.05
st = chay_lenh(loai="Market", trail=TRAIL)
kq = st.ket_qua

# Bộ bám đuổi tay
i_vao = khung.index.get_loc(pd.Timestamp(kq["mua_ngay"]))
muc = khung["close"].iloc[i_vao] * (1 - TRAIL)
duong_tay = []
for k in range(i_vao + 1, len(khung)):
    o, l, c = khung[["open", "low", "close"]].iloc[k]
    duong_tay.append((khung.index[k].date(), muc))
    if o <= muc or l <= muc:
        ngay_ra_tay, gia_ra_tay = khung.index[k].date(), min(o, muc)
        break
    muc = max(muc, c * (1 - TRAIL))

muc_bt = pd.Series(dict(st.muc_dung))  # mức ghi ở phiên t áp cho phiên t+1
muc_tay = pd.Series(dict(duong_tay)).shift(-1).dropna()  # mức áp cho phiên t+1, xếp theo phiên t
print(f"Mua {SO_CP:,} cp ngày {kq['mua_ngay']:%d/%m/%Y} giá {kq['mua_gia']:,.0f}")
print(f"Bán StopTrail ngày {kq['ban_ngay']:%d/%m/%Y} giá {kq['ban_gia']:,.2f} · tay: {ngay_ra_tay:%d/%m/%Y} giá {gia_ra_tay:,.2f}")
print(f"Giữ {len(duong_tay)} phiên · mức dừng: {muc_bt.iloc[0]:,.0f} → cao nhất {muc_bt.max():,.0f}")
print(f"Mức dừng từng phiên — lệch tối đa giữa backtrader và tay: {(muc_bt.reindex(muc_tay.index) - muc_tay).abs().max():.6f} VND")
print(f"Lãi/lỗ vòng này trước phí: {(kq['ban_gia'] - kq['mua_gia']) * SO_CP:+,.0f} VND")

Mua 1,000 cp ngày 02/03/2022 giá 24,060
Bán StopTrail ngày 11/03/2022 giá 24,852.00 · tay: 11/03/2022 giá 24,852.00
Giữ 7 phiên · mức dừng: 22,810 → cao nhất 24,852
Mức dừng từng phiên — lệch tối đa giữa backtrader và tay: 0.000000 VND
Lãi/lỗ vòng này trước phí: +792,000 VND


In [13]:
doan = khung.iloc[i_vao - 5 : khung.index.get_loc(pd.Timestamp(kq["ban_ngay"])) + 6]
ve = pd.concat(
    [
        pd.DataFrame({"date": doan.index, "gia": doan["close"] / 1_000, "duong": "Giá đóng cửa"}),
        pd.DataFrame({"date": pd.to_datetime(list(dict(duong_tay))), "gia": [m / 1_000 for _, m in duong_tay], "duong": "Mức dừng (áp cho phiên này)"}),
    ]
)
fig = duong(
    ve,
    x="date",
    y="gia",
    theo="duong",
    tieu_de=f"{MA} — lệnh bán StopTrail {TRAIL:.0%}",
    phu_de="Mức dừng chỉ đi lên; lệnh khớp ở phiên đầu tiên giá chạm mức",
    nhan_y="nghìn VND",
)
fig.add_scatter(x=[kq["mua_ngay"]], y=[kq["mua_gia"] / 1_000], mode="markers", name="mua", marker=dict(color=TANG, size=12, symbol="triangle-up"))
fig.add_scatter(x=[kq["ban_ngay"]], y=[kq["ban_gia"] / 1_000], mode="markers", name="bán", marker=dict(color=GIAM, size=12, symbol="triangle-down"))
fig

⚠️ Hai giới hạn của mô hình này trên thị trường Việt Nam:

- **Giá khớp không nằm trên bước giá.** Mức dừng là 95% của một giá đóng cửa,
  ra số lẻ tới phần trăm đồng; lệnh thật phải làm tròn về bước 10/50/100 đồng.
- **Sàn HOSE không có lệnh dừng.** `Stop` và `StopTrail` ở đây là lệnh điều kiện
  của công ty chứng khoán — chúng được *gửi* lên sàn thành lệnh thường khi điều
  kiện thoả, nên khớp thật có thể muộn hơn hoặc xấu hơn. Phiên giảm sàn trắng
  bên thì không khớp được gì — notebook `83` sẽ chặn trường hợp đó.

## 7 · Cạm bẫy: cheat-on-close

Tới giờ mọi lệnh Market đều khớp ở **giá mở cửa phiên sau** — đúng, vì tín hiệu
chỉ biết được khi phiên đóng cửa. backtrader có tuỳ chọn
`cerebro.broker.set_coc(True)` (*cheat-on-close*) để khớp lệnh ở **giá đóng cửa
của chính phiên tín hiệu**. Tên của nó đã thú nhận: đó là gian lận — mua ở cái
giá mà bạn chỉ biết sau khi nó không còn mua được nữa.

Đây chính là lỗi thứ hai của notebook
[`34`](../03-phan-tich-ky-thuat/34_backtest_chien_luoc.ipynb) ("khớp lệnh cùng
phiên tín hiệu"), giờ ở dạng một dòng cấu hình.

In [14]:
def cagr_tu(gia_tri_cuoi, so_phien):
    return (gia_tri_cuoi / VON) ** (252 / so_phien) - 1


ket_coc = {}
for coc in [False, True]:
    cerebro = cerebro_moi()
    cerebro.broker.set_coc(coc)
    cerebro.addstrategy(GiaoCatSMA)
    s = cerebro.run()[0]
    lenh = pd.DataFrame(s.nk_lenh).query("trang_thai == 'Completed'")
    ket_coc[coc] = {
        "lệnh mua đầu: phiên tín hiệu": f"{lenh['ngay_dat'].iloc[0]:%d/%m/%Y}",
        "lệnh mua đầu: phiên khớp": f"{lenh['ngay'].iloc[0]:%d/%m/%Y}",
        "giá khớp": f"{lenh['gia'].iloc[0]:,.0f}",
        "giá đóng cửa phiên tín hiệu": f"{lenh['pclose_luc_dat'].iloc[0]:,.0f}",
        "số lệnh khớp": len(lenh),
        "giá trị cuối kỳ": f"{cerebro.broker.getvalue():,.0f}",
        "CAGR": cagr_tu(cerebro.broker.getvalue(), len(khung)),
    }
bang_coc = pd.DataFrame(ket_coc).rename(columns={False: "đúng: mở cửa phiên sau", True: "SAI: set_coc(True)"})
print(f"Chênh CAGR do cheat-on-close trên {MA}: {(ket_coc[True]['CAGR'] - ket_coc[False]['CAGR']) * 100:+.2f} điểm %/năm")
bang_coc.loc["CAGR"] = bang_coc.loc["CAGR"].map("{:+.2%}".format)
bang_coc

Chênh CAGR do cheat-on-close trên HPG: -1.39 điểm %/năm


,đúng: mở cửa phiên sau,SAI: set_coc(True)
lệnh mua đầu: phiên tín hiệu,01/03/2022,01/03/2022
lệnh mua đầu: phiên khớp,02/03/2022,01/03/2022
giá khớp,"24,060","24,010"
giá đóng cửa phiên tín hiệu,"24,010","24,010"
số lệnh khớp,28,28
giá trị cuối kỳ,"96,744,175","90,216,518"
CAGR,-0.67%,-2.06%


Bản `coc` khớp **ngay trong phiên tín hiệu**, ở giá **đúng bằng** giá đóng cửa
của phiên đó. Một mã là một mẫu quá nhỏ để kết luận về độ lớn, nên đo trên
15 mã thanh khoản cao với hai chiến lược khác tính chất: giao cắt SMA (theo xu
hướng, mua sau phiên tăng) và RSI(2) (bắt đáy, mua sau phiên giảm mạnh).

In [15]:
RO = ["HPG", "FPT", "VNM", "MWG", "VCB", "TCB", "SSI", "MSN", "VHM", "GAS", "DIG", "VND", "PDR", "DXG", "GEX"]
gia_ro = bt_vn.bo_phien_chua_dong(client.eod.stock.ohlcv(RO, start=lui_ngay(HOM_NAY, nam=5)), client)


class Rsi2(GiaoCatSMA):
    """Mua khi RSI(2) < 10 và giá trên SMA100, bán khi RSI(2) > 70. Dùng lại nhật ký của GiaoCatSMA."""

    def __init__(self):
        super().__init__()
        self.rsi = bt.ind.RSI(self.data.close, period=2, safediv=True)
        self.sma100 = bt.ind.SMA(self.data.close, period=100)

    def next(self):
        if self.lenh_cho:
            return
        if not self.position and self.rsi[0] < 10 and self.data.close[0] > self.sma100[0]:
            self.lenh_cho = self.buy(size=int(self.broker.getcash() * self.p.ty_le / self.data.close[0]))
        elif self.position and self.rsi[0] > 70:
            self.lenh_cho = self.close()


def chay_ma(ma, chien_luoc, coc):
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(bt_vn.feed_tu_finlens(gia_ro, ma=ma))
    cerebro.broker.setcash(VON)
    cerebro.broker.setcommission(commission=PHI)
    cerebro.broker.set_coc(coc)
    cerebro.addstrategy(chien_luoc)
    s = cerebro.run()[0]
    lenh = pd.DataFrame(s.nk_lenh).query("trang_thai == 'Completed'")
    so_phien = int((gia_ro["symbol"] == ma).sum())
    return cagr_tu(cerebro.broker.getvalue(), so_phien), lenh


dong, khe = [], []
for ten, cl in [("Giao cắt SMA 20/50", GiaoCatSMA), ("RSI(2) bắt đáy", Rsi2)]:
    for ma in RO:
        cagr_dung, lenh = chay_ma(ma, cl, False)
        cagr_coc, _ = chay_ma(ma, cl, True)
        dong.append({"chiến lược": ten, "mã": ma, "CAGR đúng": cagr_dung, "CAGR coc": cagr_coc, "số lệnh": len(lenh)})
        # khe qua đêm: giá mở cửa phiên khớp so với giá đóng cửa phiên tín hiệu
        khe.append(lenh.assign(chien_luoc=ten, khe=lenh["gia"] / lenh["pclose_luc_dat"] - 1))

coc_ro = pd.DataFrame(dong)
coc_ro["chênh (điểm %)"] = (coc_ro["CAGR coc"] - coc_ro["CAGR đúng"]) * 100
tom = coc_ro.groupby("chiến lược").agg(
    CAGR_dung_tb=("CAGR đúng", "mean"),
    CAGR_coc_tb=("CAGR coc", "mean"),
    chenh_tb=("chênh (điểm %)", "mean"),
    chenh_tuyet_doi_tb=("chênh (điểm %)", lambda x: x.abs().mean()),
    chenh_nho_nhat=("chênh (điểm %)", "min"),
    chenh_lon_nhat=("chênh (điểm %)", "max"),
    so_ma_coc_dep_hon=("chênh (điểm %)", lambda x: int((x > 0).sum())),
)
tom.style.format(
    {"CAGR_dung_tb": "{:+.2%}", "CAGR_coc_tb": "{:+.2%}", "chenh_tb": "{:+.2f}", "chenh_tuyet_doi_tb": "{:.2f}", "chenh_nho_nhat": "{:+.2f}", "chenh_lon_nhat": "{:+.2f}"}
)

,CAGR_dung_tb,CAGR_coc_tb,chenh_tb,chenh_tuyet_doi_tb,chenh_nho_nhat,chenh_lon_nhat,so_ma_coc_dep_hon
chiến lược,,,,,,,
Giao cắt SMA 20/50,-0.18%,+0.29%,+0.46,1.22,-1.39,+4.19,8
RSI(2) bắt đáy,+0.25%,-0.72%,-0.97,1.45,-4.34,+1.89,4


In [16]:
bar_ngang(
    coc_ro[coc_ro["chiến lược"] == "Giao cắt SMA 20/50"].assign(nhan=lambda d: d["mã"]),
    nhan="nhan",
    gia_tri="chênh (điểm %)",
    tieu_de="Cheat-on-close làm CAGR đổi bao nhiêu — giao cắt SMA 20/50",
    phu_de="CAGR khi khớp ở giá đóng cửa phiên tín hiệu trừ CAGR khi khớp ở giá mở cửa phiên sau · 5 năm",
    nhan_x="điểm % mỗi năm",
    dinh_dang_nhan="{:+.2f}",
)

Kết quả **không** phải "cheat-on-close luôn làm đẹp đường vốn". Chiều lệch đổi
theo mã và theo chiến lược: với giao cắt SMA, bản `coc` đẹp hơn ở 8/15 mã; với
RSI(2), chỉ 4/15 mã. Độ lớn mới là điều đáng sợ: chênh tuyệt đối trung bình
**1,22 và 1,45 điểm %/năm**, trong khi CAGR trung bình của chính hai chiến lược
chỉ là **−0,18% và +0,25%**. Sai số do một dòng cấu hình lớn gấp nhiều lần thứ
bạn đang đo.

Vì sao chiều lệch đổi? Toàn bộ khác biệt giữa hai bản là **khe qua đêm**: giá mở
cửa phiên khớp so với giá đóng cửa phiên tín hiệu. Bản `coc` mua và bán ở giá
đóng cửa nên không bao giờ chịu khe đó.

In [17]:
khe = pd.concat(khe)
bang_khe = (
    khe.groupby(["chien_luoc", "chieu"])["khe"]
    .agg(so_lenh="size", khe_tb="mean", khe_trung_vi="median")
    .assign(khe_tb=lambda d: d["khe_tb"] * 100, khe_trung_vi=lambda d: d["khe_trung_vi"] * 100)
)
bang_khe.style.format({"khe_tb": "{:+.3f}%", "khe_trung_vi": "{:+.3f}%"})

Khe dương ở chiều **mua** nghĩa là giá mở cửa phiên sau cao hơn giá đóng cửa
phiên tín hiệu: bản đúng phải mua đắt hơn, bản `coc` được mua rẻ. Khe dương ở
chiều **bán** thì ngược lại — bản đúng bán được giá cao hơn. Hiệu ứng ròng mỗi
vòng mua-bán xấp xỉ `khe bán − khe mua`, nhân với số vòng mỗi năm.

Bảng khe giải thích đúng chiều lệch ở trên:

- **Giao cắt SMA** mua sau một chuỗi tăng, và giá thường mở cửa cao hơn nữa:
  khe mua trung bình **+0,19%**, khe bán gần 0 → bản `coc` né được khe mua, nên
  trung bình đẹp hơn.
- **RSI(2)** bán sau một nhịp hồi mạnh, và giá thường mở cửa cao hơn nữa: khe bán
  trung bình **+0,28%**, khe mua gần 0 → bản `coc` bán sớm một đêm và mất phần
  đó, nên trung bình xấu đi.

Cả hai đều là cùng một sai lầm. Nó không thiên về phía lạc quan hay bi quan — nó
thay **đúng phần lợi nhuận qua đêm** của mỗi lệnh bằng con số 0.

Đặt cạnh notebook `34`: dòng "SAI" ở mục 7 của nó chính là `set_coc(True)` viết
bằng ma trận — khớp ở giá đóng cửa phiên tín hiệu — và cho CAGR cao hơn khớp ở
giá mở cửa phiên sau 0,65 điểm % mỗi năm. Phiên bản trước của `34` còn gọi cách
khớp đó là bản đúng, vì nó tính lợi suất "từ phiên `t+1`".

**Luật cho cả Track 8: không bao giờ bật `set_coc(True)`** khi tín hiệu tính từ
giá đóng cửa. Muốn mô hình hoá lệnh ATC thì tín hiệu phải tính từ dữ liệu
*trước* 14:30 — tức dữ liệu intraday, chuyện của notebook `88`.

## Tổng kết

| Khái niệm | Điều cần nhớ |
|---|---|
| `close[0]`, `close[-1]` | chỉ số tương đối so với phiên đang xử lý |
| `close[1]` | **tương lai** — với `preload=True` mặc định thì không lỗi; chạy thử `preload=False` để bắt |
| `__init__` | chỉ báo và điều kiện cố định, tính cả chuỗi một lượt |
| `next` | quyết định theo trạng thái; chỉ chạy sau `minperiod` phiên |
| `prenext` | các phiên khởi động; quan trọng khi nhiều mã |
| `notify_order` | `Margin` là tín hiệu bị bỏ lỡ lặng lẽ — luôn ghi nhật ký |
| `notify_trade` | lãi/lỗ theo vòng, trước và sau phí |
| `Market` | giá mở cửa phiên sau — **đúng** |
| `Limit` / `Stop` / `StopTrail` | khớp theo OHLC phiên sau, kiểm được bằng tay |
| `set_coc(True)` | **không dùng** — xoá phần lãi/lỗ qua đêm; đổi CAGR nhiều hơn cả chính CAGR |

---

**Tiếp theo:** [`83_luat_choi_viet_nam.ipynb`](83_luat_choi_viet_nam.ipynb) —
thuế bán, lô 100, biên độ trần/sàn, T+2: những thứ backtrader mặc định không biết.